# **Datalab II Sprint 3 2025 - 2026**
**Plaats:** De Haagse Hogenschool, ADS & AI<br>
**Auteurs:** M. Kilinc, D. Hoogenbosch, J. Wolthuis, S. Slingerland, L. van Hamersveld<br>
**Groep:** B2 <br>
**Coach:** Onur Tezel <br>
**Datum:** 31/03/2026


| Naam  | Studentnummer |
|-------|---------------|
| Lucas | 25076116      |
| Sandro| 25154370      |
| Mehmet| 25135007      |
| Julius| 25090216      |
| Dylan | 25118498      |
---

## **Inhoudsopgave**
1. [Imports & Configuratie](#1)
2. [Data inladen](#2)
3. [Data Exploration met SQL](#3)
4. [Onderzoeken Teameigenschappen](#4)

---
<a id='1'></a>
## **1. Imports & Configuratie**

In [1]:
# Imports
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
from scipy.stats import pearsonr, f_oneway

In [2]:
# Via deze functie in pandas zorgen we ervoor dat we de DataFrames volledig kunnen weergeven in de cell
pd.set_option('display.max_rows', 200)

In [3]:
# Herbruikbare functie om een SQL-query uit te voeren en het resultaat te tonen
def voer_query_uit(query, conn, toon=True):
    """Voert een SQL-query uit en geeft het resultaat terug als DataFrame.
    
    Parameters:
        query (str): De SQL-query om uit te voeren.
        conn: De database connectie.
        toon (bool): Als True wordt het DataFrame getoond. Standaard True.
    """
    df = pd.read_sql(query, conn)
    if toon:
        display(df)
    return df

---
<a id='2'></a>
## **2. Data inladen**
In de onderstaande kolommen laden we de data op dezelfde manier als de vorige keer, namelijk via de Kaggle API. <br> 
Hieronder vind je een uitleg over het gebruik en een referentie.



### **2.1 Gebruik van de Kaggle API** 
We gebruiken de Kaggle API om de data lokaal op te slaan en dit hoeft slechts eenmalig te gebeuren.<br>Download de API-sleutel uit onze GitHub-repository en plaats deze in de map C:\Users\<Jouw_gebruikersnaam>\.kaggle. <br>

Raadpleeg voor verdere instructies de officiële documentatie op https://www.kaggle.com/docs/api.<br>
Verwijder eventueel het # in het script, voer de pip install en de data-import uit en je bent klaar om de data lokaal te gebruiken.


In [4]:
# %pip install kaggle 
import kaggle 
kaggle.api.authenticate()
kaggle.api.dataset_download_files("hugomathien/soccer", path='.', unzip=True)

Dataset URL: https://www.kaggle.com/datasets/hugomathien/soccer


In [5]:
# Maak een verbinding met de SQLite-database "database.sqlite"
conn = sqlite3.connect("database.sqlite")

# Voer een SQL-query uit om alle gegevens uit de tabel 'Match' op te halen, het resultaat wordt ingelezen als pandas DataFrame
df_match = pd.read_sql("""

    SELECT *
        FROM Match

    """,
    conn)

---
<a id='4'></a>
## **4. Onderzoeken Teameigenschappen**

---
### 4.1 Teameigenschappen samenvoegen

In [6]:
# query om teameigenschappen te selecteren en te groeperen op teamnaam
query = """
SELECT
    T.team_long_name AS Team,
    AVG(TA.buildUpPlaySpeed)            AS buildUpPlaySpeed,
    AVG(TA.buildUpPlayDribbling)        AS buildUpPlayDribbling,
    AVG(TA.buildUpPlayPassing)          AS buildUpPlayPassing,
    AVG(TA.chanceCreationPassing)       AS chanceCreationPassing,
    AVG(TA.chanceCreationCrossing)      AS chanceCreationCrossing,
    AVG(TA.chanceCreationShooting)      AS chanceCreationShooting,
    AVG(TA.defencePressure)             AS defencePressure,
    AVG(TA.defenceAggression)           AS defenceAggression,
    AVG(TA.defenceTeamWidth)            AS defenceTeamWidth
FROM Team_Attributes AS TA
JOIN Team AS T ON TA.team_api_id = T.team_api_id
GROUP BY T.team_long_name;
"""

# Opslaan data uit query in DataFrame
df_attrs = voer_query_uit(query, conn, toon=False)

# Query om team punten te berekenen
punten_query = """
SELECT
    T.team_long_name AS Team,
    COUNT(*) as Wedstrijden,
    SUM(CASE
        WHEN M.home_team_api_id = T.team_api_id AND M.home_team_goal > M.away_team_goal THEN 3
        WHEN M.away_team_api_id = T.team_api_id AND M.away_team_goal > M.home_team_goal THEN 3
        WHEN M.home_team_api_id = T.team_api_id AND M.home_team_goal = M.away_team_goal THEN 1
        WHEN M.away_team_api_id = T.team_api_id AND M.home_team_goal = M.away_team_goal THEN 1
        ELSE 0
    END) as Punten
FROM Team AS T
JOIN Match AS M ON (M.home_team_api_id = T.team_api_id OR M.away_team_api_id = T.team_api_id)
GROUP BY T.team_long_name
ORDER BY Punten DESC;
"""

# Opslaan punten data in DataFrame
df_2 = voer_query_uit(punten_query, conn, toon=False)

# Samenvoegen met het punten DataFrame op teamnaam
df_merged = pd.merge(df_2, df_attrs, on='Team', how='inner')

# DataFrame met alle teameigenschappen en punten per team
display(df_merged.head(10))

# Print grote van de samengevoegde DataFrame
print(f'Samengevoegd DataFrame: {len(df_merged)} rijen, {len(df_merged.columns)} kolommen')

,Team,Wedstrijden,Punten,buildUpPlaySpeed,buildUpPlayDribbling,buildUpPlayPassing,chanceCreationPassing,chanceCreationCrossing,chanceCreationShooting,defencePressure,defenceAggression,defenceTeamWidth
0,FC Barcelona,304,745,35.833333,35.0,34.000000,45.166667,33.333333,53.000000,64.333333,54.500000,66.500000
1,Real Madrid CF,304,720,50.666667,55.5,38.666667,67.500000,53.833333,69.000000,52.000000,52.166667,63.500000
2,Celtic,304,704,61.333333,50.5,57.333333,55.666667,60.833333,60.833333,53.333333,54.333333,62.166667
3,Manchester United,304,633,51.833333,38.0,45.833333,50.000000,63.166667,53.833333,45.000000,48.000000,54.833333
4,Juventus,301,633,45.833333,41.0,32.833333,56.666667,60.500000,65.666667,44.000000,57.333333,41.166667
5,FC Bayern Munich,272,623,48.666667,29.0,35.166667,38.500000,40.666667,49.666667,57.166667,48.833333,55.166667
6,FC Basel,286,604,55.000000,63.0,44.833333,60.833333,63.833333,52.500000,47.666667,57.500000,54.500000
7,Ajax,272,602,35.166667,41.5,33.833333,50.666667,58.500000,49.666667,59.833333,53.833333,54.333333
8,Paris Saint-Germain,304,601,46.833333,58.5,45.166667,53.833333,54.333333,54.666667,54.500000,53.833333,58.500000
9,Chelsea,304,598,61.666667,46.5,45.000000,49.833333,55.166667,61.666667,40.833333,50.333333,44.833333


Samengevoegd DataFrame: 285 rijen, 12 kolommen


In [7]:
# Database connection is already established above as 'conn'

---

In [ ]:
def bereken_speler_gemiddelden(connection, seizoen=None, sorteer_op='avg_potential'):
    """
    Berekent gemiddelde attributen en identificeert keepers op basis van gk_reflexes.
    Optioneel filteren op een specifiek seizoen (bijv. '2008/2009').
    Optioneel sorteren op een specifieke kolom (standaard 'avg_potential').
    """
    query = """
    SELECT
        P.player_name AS Player,
        AVG(PA.overall_rating) AS avg_overall_rating,
        AVG(PA.potential) AS avg_potential,
        AVG(PA.finishing) AS avg_finishing,
        AVG(PA.dribbling) AS avg_dribbling,
        AVG(PA.sprint_speed) AS avg_sprint_speed,
        AVG(PA.stamina) AS avg_stamina,
        AVG(PA.gk_reflexes) AS avg_gk_reflexes
    FROM Player_Attributes AS PA
    JOIN Player AS P ON PA.player_api_id = P.player_api_id
    """
    
    if seizoen:
        # Parse seizoen, bijv. '2008/2009' -> start_year=2008, end_year=2009
        start_year, end_year = map(int, seizoen.split('/'))
        start_date = f"{start_year}-08-01"
        end_date = f"{end_year}-05-31"
        query += f" WHERE PA.date BETWEEN '{start_date}' AND '{end_date}'"
    
    query += " GROUP BY P.player_name;"
    
    # Data inladen
    df = pd.read_sql(query, connection)

    # Functie om rol te bepalen: als reflexen veel hoger zijn dan finishing, is het een keeper
    # Een drempelwaarde van 30-40 voor reflexen werkt meestal perfect voor deze dataset
    def bepaal_rol(row):
        gk_reflexes = row['avg_gk_reflexes'] if pd.notna(row['avg_gk_reflexes']) else 0
        finishing = row['avg_finishing'] if pd.notna(row['avg_finishing']) else 0
        if gk_reflexes > 30 and gk_reflexes > finishing:
            return 'Keeper'
        else:
            return 'Veldspeler'

    df['Rol'] = df.apply(bepaal_rol, axis=1)

    # Sorteer op de opgegeven kolom
    df = df.sort_values(by=sorteer_op, ascending=False)

    return df

# Uitvoeren
df_resultaat = bereken_speler_gemiddelden(conn, seizoen='2014/2015')

display(df_resultaat.head(50))

,Player,avg_overall_rating,avg_potential,avg_finishing,avg_dribbling,avg_sprint_speed,avg_stamina,avg_gk_reflexes,Rol
3932,Lionel Messi,93.000000,95.000000,94.000000,96.000000,90.000000,77.000000,8.0,Veldspeler
1290,Cristiano Ronaldo,92.000000,92.000000,95.000000,93.000000,94.000000,89.000000,11.0,Veldspeler
2926,James Rodriguez,86.000000,92.000000,80.000000,86.000000,79.000000,73.000000,14.0,Veldspeler
2345,Gareth Bale,87.000000,91.000000,81.000000,87.000000,95.000000,90.000000,6.0,Veldspeler
5043,Neymar,86.000000,91.000000,85.000000,94.000000,89.000000,86.000000,11.0,Veldspeler
4356,Mario Goetze,85.636364,90.454545,81.363636,90.000000,76.090909,71.000000,10.0,Veldspeler
4065,Luis Suarez,89.000000,90.250000,91.000000,90.000000,79.000000,86.000000,37.0,Veldspeler
5379,Paul Pogba,83.250000,90.125000,70.000000,86.000000,77.000000,87.000000,3.0,Veldspeler
639,Arjen Robben,90.000000,90.000000,85.000000,93.000000,93.000000,78.000000,15.0,Veldspeler
1825,Eden Hazard,88.000000,90.000000,83.000000,92.800000,87.000000,74.000000,8.0,Veldspeler


In [ ]:
# Functie om beschikbare seizoenen op te halen uit de Match tabel
def haal_beschikbare_seizoenen(connection):
    query = "SELECT DISTINCT season FROM Match ORDER BY season;"
    df = pd.read_sql(query, connection)
    return df['season'].tolist()

# Voorbeeld: beschikbare seizoenen
seizoenen = haal_beschikbare_seizoenen(conn)
print("Beschikbare seizoenen:", seizoenen)